# RepairCalc: обучение и оценка work-type моделей
Последовательно выполните все ячейки. Метрики на weak labels являются только техническим agreement; итоговая оценка требует `review_status=reviewed/approved`.

## 1. Настройка окружения

In [ ]:
!pip -q install pandas numpy scikit-learn joblib catboost matplotlib
# Только для embeddings_logreg; Colab уже содержит совместимый PyTorch:
# !pip -q install sentence-transformers

## 2. Проверка GPU

In [ ]:
import os, subprocess, torch
print('CUDA доступна:', torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))

## 3. Доступ к репозиторию
Для публичного репозитория достаточно `git clone`. Для private repo создайте Colab Secret `GITHUB_TOKEN`, считайте его через `google.colab.userdata.get` и сформируйте URL только в памяти. Никогда не записывайте token в notebook или Git.

In [ ]:
from pathlib import Path
REPO_URL = 'https://github.com/OWNER/REPOSITORY.git'  # замените
REPO_DIR = Path('/content/IYM')
if not REPO_DIR.exists():
    !git clone $REPO_URL $REPO_DIR
os.chdir(REPO_DIR)
print(Path.cwd())

## 4. Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/RepairCalc')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

## 5–7. Пути и выбор эксперимента

In [ ]:
import sys
sys.path.insert(0, str(REPO_DIR / 'ml_workspace'))
from ml_api import (TrainingConfig, load_data, prepare_features, train, validate, evaluate,
                    save_model, load_model, predict, ProductInput)
from ml_api.reporting import save_evaluation_outputs
EXPERIMENT = 'tfidf_logreg'  # tfidf_logreg | catboost_structured | embeddings_logreg
LABEL_POLICY = 'reviewed_only'  # для pilot: weak_only или reviewed_plus_weak
USE_GPU = torch.cuda.is_available() and EXPERIMENT in {'catboost_structured', 'embeddings_logreg'}
CLEAN_PATH = REPO_DIR / 'ml_workspace/data/products_clean.csv'
WEAK_PATH = REPO_DIR / 'ml_workspace/data/products_multidim_weak_labeled.csv'
REVIEW_PATH = DRIVE_ROOT / 'manual_review_500.csv'
TAXONOMY_PATH = REPO_DIR / 'ml_workspace/taxonomy/work_types_draft.csv'
SPLIT_PATH = DRIVE_ROOT / 'work_type_split_manifest.csv'
ARTIFACT_DIR = Path('/content/artifacts') / EXPERIMENT
print(EXPERIMENT, LABEL_POLICY, USE_GPU)

## 8–9. Загрузка verified annotations и общий split

In [ ]:
data = load_data(clean_path=CLEAN_PATH, weak_labels_path=WEAK_PATH, reviewed_paths=[REVIEW_PATH],
                 split_manifest_path=SPLIT_PATH, taxonomy_path=TAXONOMY_PATH,
                 taxonomy_version='work_types_draft_v2', label_policy=LABEL_POLICY, random_seed=20261023)
print(data.frame.groupby(['split', 'label_source']).size())
for split, group in data.frame.groupby('split'):
    support = {label: sum(label in labels for labels in group.labels) for label in data.class_names}
    print(split, support)

## 10. Подготовка признаков и обучение

In [ ]:
config = TrainingConfig(experiment=EXPERIMENT, random_seed=20261023, use_gpu=USE_GPU)
features = prepare_features(data, config, cache_dir=DRIVE_ROOT / 'embedding_cache')
model = train(features, config)

## 11–12. Validation thresholds и финальная test evaluation
Thresholds выбираются только на validation. Test не используется для настройки.

In [ ]:
validation = validate(model, features)
metrics = evaluate(model, features, split='test')
print('Provenance:', metrics.label_provenance)
print({k: v for k, v in metrics.metrics.items() if k != 'per_class'})

## 13. Визуализации и error analysis

In [ ]:
save_evaluation_outputs(metrics, model, Path('/content/evaluation'))
from IPython.display import display, Image
display(Image('/content/evaluation/per_class_scores.png'))
display(Image('/content/evaluation/class_support.png'))
display(Image('/content/evaluation/label_count_distribution.png'))
display(__import__('pandas').read_csv('/content/evaluation/errors.csv').head(30))

## 14–15. Сохранение и копирование в Drive

In [ ]:
import shutil
save_model(model, ARTIFACT_DIR, validation, metrics)
save_evaluation_outputs(metrics, model, ARTIFACT_DIR / 'evaluation')
DRIVE_ARTIFACT = DRIVE_ROOT / 'artifacts' / EXPERIMENT
if DRIVE_ARTIFACT.exists(): shutil.rmtree(DRIVE_ARTIFACT)
shutil.copytree(ARTIFACT_DIR, DRIVE_ARTIFACT)
print(DRIVE_ARTIFACT)

## 16–17. Загрузка и inference

In [ ]:
loaded_model = load_model(DRIVE_ARTIFACT, expected_taxonomy_version='work_types_draft_v2', trusted=True)
new_products = [ProductInput(product_id=None, name='краска для стен влагостойкая белая',
                             source_category='Краски для стен и потолков',
                             description='Интерьерная краска для влажных помещений', price=1200)]
predictions = predict(loaded_model, new_products)
predictions